# Langchain Exprsession Language Basics
- Langchain Exprsession Language is that any two runnables can be "chained" together into sequences.
- The output of the previous runnable's`.invoke()` call is passed as input to the next **runnable**.
- This can be done using the pipe operator (`|`), or the more explicit `.pipe()` method, which does the same thing.

- Types of LCEL Chains:
  - Sequential Chain
  - Parallel Chain
  - Router Chain
  - Chain Runnables
  - Custom Chain (Runnable Sequence)

In [1]:
from dotenv import load_dotenv

load_dotenv('../../.env')


True

### Sequential LCEL Chain
Sequential Langchain Exprsession Language Chain

In [5]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate

base_url = os.getenv('OLLAMA_BASE_URL')
model = os.getenv('OLLAMA_LLM_MODEL')

llm = ChatOllama(base_url=base_url, model=model)


human_template = HumanMessagePromptTemplate.from_template('Tell me about the {topic} in {points} points.')
system_template = SystemMessagePromptTemplate.from_template('You are {school} teacher. You answer in short sentences.')

template = ChatPromptTemplate([system_template, human_template])

We do not need to call `template.invoke()` and `llm.invoke()` separetedly. We may create a **chain** and invoke them at once, using the pipe operator.

In [3]:
from langchain.messages import AIMessage
chain = template | llm
response: AIMessage = chain.invoke({'school': 'phd', 'topic': 'solar system', 'points': 3})
print(response.content) 

1. The Sun contains the majority of the mass.
2. Planets orbit in elliptical paths.
3. Gravity governs the motion.


This is possible because `template` and `llm` are subclasses of `Runnable`. They implement the invoke method, and may be chained to be called at once using the `|` (pipe) operator.

We may also chain the `response.content` call using `StrOutputParser`:

In [4]:
from langchain_core.output_parsers import StrOutputParser
chain = template | llm | StrOutputParser()
response: str = chain.invoke({'school': 'phd', 'topic': 'solar system', 'points': 3})
print(response)

1. The Sun accounts for 99.8% of the solar system's mass.
2. Planets orbit the Sun in elliptical paths around it.
3. Gravity governs the orbital motion of all celestial bodies.


### Chaining Chains
- We can even combine this chain with more runnables to create another chain.
  

In [3]:
# This is the chain we already have. Let's call it chain_1_fact
chain_1_fact = chain
chain_1_fact

NameError: name 'chain' is not defined

In [ ]:
# This is the response we got from the execution of the chain 1
response

"1. The Sun accounts for 99.8% of the solar system's mass.\n2. Planets orbit the Sun in elliptical paths around it.\n3. Gravity governs the orbital motion of all celestial bodies."

Now we are creating it another capable of analyze a text.

In [9]:
analysis_prompt = ChatPromptTemplate.from_template('''Analyze the following text: {text}.
                                                   You need to tell me how difficult it is to understand.
                                                   Answer in one sentence only.''')
chain_2_analysis = analysis_prompt | llm | StrOutputParser()

Now we may chain the two chains in one single invoke:

In [ ]:
composed_chain = { 'text': chain_1_fact } | chain_2_analysis
output = composed_chain.invoke({'school': 'elementary', 'topic': 'solar system', 'points': 3})
output

'This text is extremely easy to understand due to its simple vocabulary and clear concepts.'

In [ ]:
phd_output = composed_chain.invoke({'school': 'phd', 'topic': 'solar system', 'points': 3})
phd_output

'The text is relatively simple and easy to understand as it relies on accessible language and clear scientific terminology.'

## Parallel LCEL Chain
- Parallel chains are used to run multiple runnables in parallel
- It is useful when the chains are independent
- The final return is a dict with the result of each value under its appropriate key


We are now going to invoke two chains in parallel: 
- fact_chain: let's use the elements that we already have


In [7]:
from langchain_core.output_parsers import StrOutputParser
fact_chain =  template | llm | StrOutputParser()

- poem_chain: let's create a new one

In [8]:
human_template = HumanMessagePromptTemplate.from_template('Write a poem on {topic} in {num_of_sentences} lines.')

template = ChatPromptTemplate([system_template, human_template])

poem_chain = template | llm | StrOutputParser()




In [9]:
from langchain_core.runnables import RunnableParallel
parallel_chain = RunnableParallel(fact=fact_chain, poem=poem_chain)

parellel_output: dict = parallel_chain.invoke({'school': 'elementary', 'topic': 'solar system', 'points': 3, 'num_of_sentences': 5})

print(f"--- Fact --- \n{parellel_output['fact']}")
print('\n\n')
print(f"--- Poem --- \n{parellel_output['poem']}")

--- Fact --- 
1. The sun shines in the middle.
2. Planets go around it.
3. Earth has water and life.



--- Poem --- 
The Sun is hot and bright.
The Moon is small and white.
The Earth spins around us.
The planets orbit round him.
Mars is red like dirt.
